<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_10_Large_Language_Models_with_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 10: Training Large Language Models from Scratch (GPT-2)



Instructor: John Sipple

**Frameworks:** JAX, Optax, Equinox

**Important Notes:**

* Training large language models is exceptionally resource-intensive. To successfully execute the code and avoid Out-Of-Memory (OOM) crashes in this notebook, you will need access to High-RAM environments and advanced hardware accelerators (such as A100 GPUs or TPU v2/v3/v4 pods). This tutorial works best with a Colab Pro or Colab Pro Plus subscription.

* This notebiok was written to run on the limited resources of a hosted runtime while demonstrating multiple advanced LLM techniques. However, due to these limitations, the resulting output will be very limited. In other words, with the down-sized models presented in this notebook, don't expect high-end LLM performance!

### Overview

Welcome to the capstone of sequence modeling. In this notebook, we construct a robust Decoder-Only Transformer entirely from scratch. Moving beyond the foundational "Nano-GPT" from earlier lectures, we will walk through the complete lifecycle of a modern Large Language Model (LLM). This tutorial demystifies the multi-stage pipeline used to create state-of-the-art models, providing a transparent, step-by-step JAX implementation of the four foundational pillars of LLM training.

### Key Concepts Explored:

* **Unsupervised Pre-Training:** Building the foundational statistical representation by training the model to autoregressively predict the next token across large, unstructured text corpora.
* **Supervised Fine-Tuning (SFT):** Transitioning the model from a mere "document completer" into a helpful conversational assistant by training it on high-quality, formatted instruction-response pairs.
* **Parameter-Efficient Fine-Tuning (PEFT) with LoRA:** Implementing Low-Rank Adaptation (LoRA) to freeze the massive core model weights and only train a tiny percentage of newly injected, low-rank matrices. This drastically reduces the VRAM overhead required during fine-tuning.
* **Reinforcement Learning from Human Feedback (RLHF):** Aligning the model's outputs with human preferences and safety guidelines by training a reward model and using reinforcement learning to optimize the LLM policy against it.
* **Modern Architectural Upgrades:** Exploring state-of-the-art modifications to the standard Transformer block, including replacing standard MLPs with SwiGLU gated activations for better parameter scaling, and implementing Grouped-Query Attention (GQA) to drastically reduce the KV-Cache memory footprint during generation.
* **Distributed Training Infrastructure:** Understanding how to scale beyond a single device. We explore JAX's `pjit` (sharding) and Fully Sharded Data Parallel (FSDP) mechanisms to elegantly partition massive PyTrees—including model weights and optimizer states—across multiple GPUs or TPU cores.

### Real-World Application

This tutorial provides the exact blueprint for enterprise AI teams that need to securely adapt and fine-tune domain-specific LLMs (such as specialized medical diagnosis assistants or financial contract analyzers) on highly proprietary data without passing sensitive information to external, closed-source APIs.

In [ ]:
# Install necessary libraries
!pip install -q equinox optax tiktoken requests matplotlib

import os
import requests
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx
import optax
import tiktoken
import numpy as np
import matplotlib.pyplot as plt

# Verify backend
print(f"JAX backend: {jax.default_backend().upper()}")
print(f"Available devices: {jax.devices()}")

# Utility function for plotting loss curves
def plot_losses(train_losses, test_losses, title="Training vs Test Loss"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss", color="blue")
    plt.plot(test_losses, label="Test Loss", color="orange", linestyle="--")
    plt.title(title)
    plt.xlabel("Evaluation Step")
    plt.ylabel("Cross-Entropy Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# ==========================================
# CHECKPOINTING SETUP
# ==========================================
import os
from google.colab import drive

print("Mounting Google Drive to save checkpoints safely...")
drive.mount('/content/drive')

# Create a directory in the student's Google Drive for the weights
ckpt_dir = '/content/drive/MyDrive/CSCI6366_LLM_Checkpoints2'
os.makedirs(ckpt_dir, exist_ok=True)

print(f"Checkpoints will be saved to: {ckpt_dir}")

#Part 1: Dataset Preparation

We will use the **Tiny Shakespeare** dataset and the tiktoken BPE tokenizer. Crucially, we split the data into a **90% Training Set** and a **10% Test Set** to track generalization and prevent overfitting.

In [ ]:
def download_corpus():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    filepath = "shakespeare.txt"
    if not os.path.exists(filepath):
        print("Downloading Tiny Shakespeare corpus...")
        response = requests.get(url)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(response.text)
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

text = download_corpus()
tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab

# Split the data
split_idx = int(0.9 * len(text))
train_text, test_text = text[:split_idx], text[split_idx:]

train_tokens = np.array(tokenizer.encode(train_text), dtype=np.int32)
test_tokens = np.array(tokenizer.encode(test_text), dtype=np.int32)

print(f"Vocabulary size: {vocab_size} tokens")
print(f"Train tokens: {len(train_tokens):,} | Test tokens: {len(test_tokens):,}")

def get_batch(tokens, batch_size, seq_len, key):
    max_idx = len(tokens) - seq_len - 1
    start_idxs = jrandom.randint(key, (batch_size,), 0, max_idx)
    batch = np.stack([tokens[i:i+seq_len+1] for i in start_idxs])
    return jnp.array(batch)

###Inspecting the Corpus and Batches

Before we build the model, let's look at exactly what the data looks like. The neural network will never see the English text; it will only see the integer arrays.

Notice how in the **Batching Example**, the target sequence ($Y$) is identical to the context sequence ($X$), but shifted one token into the future. This is the essence of autoregressive pre-training.

In [ ]:
# 1. Raw Text Snippet
print("========== 1. RAW TEXT EXAMPLE ==========")
print(train_text[:250])
print("=========================================\n")

# 2. Tokenization Example
print("========== 2. TOKENIZATION EXAMPLE ==========")
sample_text = "O Romeo, Romeo!"
sample_tokens = tokenizer.encode(sample_text)
print(f"Text:   '{sample_text}'")
print(f"Tokens: {sample_tokens}")
print(f"Verify: '{tokenizer.decode(sample_tokens)}'")
print("=============================================\n")

# 3. Batching Example (X -> Y Shift)
print("========== 3. BATCHING EXAMPLE (X -> Y) ==========")
# Let's pull a tiny sequence of length 8 to see how the model learns
sample_key = jrandom.PRNGKey(1337)
tiny_batch = get_batch(train_tokens, batch_size=1, seq_len=8, key=sample_key)[0]

# Inputs (all tokens except the last)
x = tiny_batch[:-1]
# Targets (all tokens except the first)
y = tiny_batch[1:]

print("Context (X):")
print(f"  Array:   {x}")
print(f"  Decoded: '{tokenizer.decode(x.tolist())}'\n")

print("Targets (Y):")
print(f"  Array:   {y}")
print(f"  Decoded: '{tokenizer.decode(y.tolist())}'")
print("==================================================")

# Part 2: Architecture and Diagnostics

We will implement a standard autoregressive Transformer. Key components:

* **Token & Positional Embeddings**: Mapping integers to dense vectors and injecting sequence order.

* **Causal Self-Attention**: Using a lower-triangular mask so token $t$ can only attend to tokens $\le t$.

* **Pre-LayerNorm**: Applying normalization before attention and MLP blocks, which is crucial for deep network stability.

We also introduce two useful diagnostic methods:

1. `render_architecture()`: Uses Equinox's elegant PyTree formatting to print the exact layout.

2. `print_parameter_counts()`: Iterates through the network leaves to calculate the memory footprint of each layer.

###A note about the positional embeddings

In the original 2017 paper "*Attention Is All You Need*", the authors indeed used fixed mathematical functions (sines and cosines) of different frequencies to encode position. However, in our tutorial (and in models like BERT, GPT-1, GPT-2, and GPT-3), we used a standard, trainable Embedding Layer.

Here is why the industry shifted to learned embeddings, and why the math is now coming back full circle.

**1. The Original Approach: Sinusoidal Encodings**

The original Transformer used fixed sine and cosine functions.

* **The Theory**: By using continuous mathematical functions, the model should theoretically be able to extrapolate—meaning if you train it on sequences of length 512, it should be able to figure out how to handle token 513 during inference because the sine wave continues predictably.

* **The Reality**: In practice, models trained with absolute sinusoidal encodings struggled to extrapolate well. When presented with sequences longer than their training data, performance degraded rapidly anyway.

**2. The OpenAI Approach: Learned Absolute Embeddings**

When researchers at OpenAI built GPT-1 (and subsequently GPT-2 and GPT-3), they made a pragmatic decision: **If the model has hundreds of millions of parameters and a massive dataset, why hand-code the math? Let the network learn the optimal positional representations itself.**

* **How it works:** An eqx.nn.Embedding(seq_len, d_model) simply initializes a matrix of random noise. Over millions of training steps, the model learns via backpropagation exactly what "position 5" means relative to "position 10".

* **The Advantage:** It is incredibly easy to implement and empirically performed slightly better or on-par with sinusoidal encodings for the exact sequence lengths it was trained on.

* **The Disadvantage (The Hard Limit):** If you initialize an embedding table of size 1024, the model literally does not have a vector for position 1025. If you feed it a prompt of length 1025, the code will crash with an out-of-bounds index error.

**3. The Modern Era: Sines and Cosines Return!**

As models grew, the need for massive context windows (like Claude's 200,000 token window) became paramount. Hard-coding a learned embedding layer with 200,000 vectors is computationally wasteful and rigid.

This led to the invention of **Rotary Positional Embeddings (RoPE)**, which is what LLaMA 3, Gemma, and Mistral use today.

RoPE brings the sines and cosines back, but instead of adding them to the token embeddings at the very beginning of the network, it uses a rotation matrix containing sines and cosines to rotate the Query and Key vectors inside the attention mechanism itself. This perfectly captures **relative distance** (e.g., token A is 4 steps away from token B) rather than just absolute position.

In [ ]:
class TransformerBlock(eqx.Module):
    attention: eqx.nn.MultiheadAttention
    mlp: eqx.nn.MLP
    ln1: eqx.nn.LayerNorm
    ln2: eqx.nn.LayerNorm

    def __init__(self, d_model, num_heads, key):
        key1, key2 = jrandom.split(key)
        self.attention = eqx.nn.MultiheadAttention(
            num_heads=num_heads, query_size=d_model,
            use_query_bias=False, use_key_bias=False,
            use_value_bias=False, use_output_bias=False, key=key1
        )
        self.mlp = eqx.nn.MLP(
            in_size=d_model, out_size=d_model, width_size=d_model * 4,
            depth=1, activation=jax.nn.gelu, key=key2
        )
        self.ln1 = eqx.nn.LayerNorm(d_model)
        self.ln2 = eqx.nn.LayerNorm(d_model)

    def __call__(self, x, mask):
        # FIX: Changed 'key=' to 'key_=' to satisfy Equinox's strict naming convention
        attn_out = self.attention(
            query=jax.vmap(self.ln1)(x),
            key_=jax.vmap(self.ln1)(x),
            value=jax.vmap(self.ln1)(x),
            mask=mask
        )
        x = x + attn_out
        mlp_out = jax.vmap(self.mlp)(jax.vmap(self.ln2)(x))
        return x + mlp_out

class LLM(eqx.Module):
    embedding: eqx.nn.Embedding
    pos_embedding: eqx.nn.Embedding
    blocks: list
    ln_f: eqx.nn.LayerNorm
    lm_head: eqx.nn.Linear

    def __init__(self, vocab_size, seq_len, d_model, num_heads, num_layers, key):
        keys = jrandom.split(key, num_layers + 2)
        self.embedding = eqx.nn.Embedding(vocab_size, d_model, key=keys[0])
        self.pos_embedding = eqx.nn.Embedding(seq_len, d_model, key=keys[1])
        self.blocks = [TransformerBlock(d_model, num_heads, key=keys[i+2]) for i in range(num_layers)]
        self.ln_f = eqx.nn.LayerNorm(d_model)
        self.lm_head = eqx.nn.Linear(d_model, vocab_size, use_bias=False, key=keys[-1])

    def __call__(self, x):
        seq_len = x.shape[0]
        mask = jnp.where(jnp.tril(jnp.ones((seq_len, seq_len))) == 0, -jnp.inf, 0.0)
        positions = jnp.arange(seq_len)
        x = jax.vmap(self.embedding)(x) + jax.vmap(self.pos_embedding)(positions)
        for block in self.blocks: x = block(x, mask)
        x = jax.vmap(self.ln_f)(x)
        return jax.vmap(self.lm_head)(x)

# --- Diagnostics ---
def render_architecture(model):
    print("========== MODEL ARCHITECTURE ==========")
    print(eqx.tree_pformat(model))
    print("========================================")

def print_parameter_counts(model):
    print("========== PARAMETER COUNTS ==========")
    def count_params(x): return x.size if isinstance(x, jax.Array) else 0

    # FIX: Replaced the deprecated jax.tree_map with jax.tree_util.tree_map
    embed_params = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(count_params, model.embedding)))
    pos_params = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(count_params, model.pos_embedding)))
    print(f"Token Embedding:      {embed_params:,}")
    print(f"Positional Embedding: {pos_params:,}")

    for i, block in enumerate(model.blocks):
        b_params = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(count_params, block)))
        print(f"Transformer Block {i}: {b_params:,}")

    head_params = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(count_params, model.lm_head)))
    total = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(count_params, model)))
    print(f"Language Model Head:  {head_params:,}")
    print(f"--------------------------------------")
    print(f"TOTAL PARAMETERS:     {total:,}")
    print("======================================")


# --- UPGRADED HYPERPARAMETERS (GPT-2 Small Scale) ---
SEQ_LEN = 256       # Quadrupled context window (was 64)
D_MODEL = 256       # Tripled representation width (was 256)
NUM_HEADS = 12      # Tripled attention heads (was 4)
NUM_LAYERS = 6     # Tripled network depth (was 4)

key = jrandom.PRNGKey(42)
model = LLM(vocab_size, SEQ_LEN, D_MODEL, NUM_HEADS, NUM_LAYERS, key)

render_architecture(model)
print_parameter_counts(model)

## The Autoregressive Generation Loop

To see the model learn, we need a function to generate text. This function takes a prompt, passes it through the model, samples the most likely next token, appends it, and repeats.

In [ ]:
def generate_text(model, prompt_str, tokenizer, max_new_tokens=30, seq_len=SEQ_LEN):
    tokens = tokenizer.encode(prompt_str)

    for _ in range(max_new_tokens):
        # Truncate to context window if necessary
        window = tokens[-seq_len:] if len(tokens) > seq_len else tokens

        # Forward pass (no gradients needed)
        logits = model(jnp.array(window))

        # Greedy decoding (take the argmax of the last token's logits)
        next_token = jnp.argmax(logits[-1])
        tokens.append(int(next_token))

    return tokenizer.decode(tokens)

# Baseline generation (Random weights)
print("--- PRE-TRAINING BASELINE GENERATION ---")
print(generate_text(model, "O Romeo, Romeo!", tokenizer, max_new_tokens=20))

In [ ]:
def sample_top_k(logits, key, temperature=0.8, top_k=40, repetition_penalty=1.2, generated_tokens=None):
    """
    Samples a token from logits using Temperature, Top-K, and Repetition Penalty.
    """
    # 1. Apply Repetition Penalty
    if generated_tokens is not None and repetition_penalty != 1.0:
        # Create a mask of tokens we've already generated
        unique_tokens = jnp.unique(jnp.array(generated_tokens))

        # JAX arrays are immutable, so we create a penalty array
        # Standard formulation: if logit > 0, divide by penalty. If < 0, multiply.
        penalties = jnp.ones_like(logits)

        # We use a simple loop here since vocab size is small,
        # but in production this would be vectorized.
        for t in unique_tokens:
            logit_val = logits[t]
            penalty_factor = jnp.where(logit_val > 0, 1.0 / repetition_penalty, repetition_penalty)
            logits = logits.at[t].multiply(penalty_factor)

    # 2. Apply Temperature
    # Higher temperature = flatter distribution (more random)
    # Lower temperature = sharper distribution (more greedy)
    logits = logits / temperature

    # 3. Apply Top-K
    # Find the threshold value for the Kth largest logit
    top_k_threshold = jnp.sort(logits)[-top_k]

    # Mask out anything below the threshold by setting to -infinity
    logits = jnp.where(logits < top_k_threshold, -jnp.inf, logits)

    # 4. Sample from the modified distribution using a PRNG key
    return jax.random.categorical(key, logits)

def generate_text_advanced(model, prompt_str, tokenizer, key, max_new_tokens=50, seq_len=SEQ_LEN,
                           temperature=0.8, top_k=40, rep_penalty=1.2):
    """
    Advanced autoregressive generation loop.
    """
    tokens = tokenizer.encode(prompt_str)

    print(prompt_str, end="") # Print prompt immediately

    for _ in range(max_new_tokens):
        # Truncate to context window
        window = tokens[-seq_len:] if len(tokens) > seq_len else tokens

        # Forward pass
        logits = model(jnp.array(window))

        # Split key for randomness
        key, subkey = jrandom.split(key)

        # Sample the next token
        next_token = sample_top_k(
            logits[-1],
            subkey,
            temperature=temperature,
            top_k=top_k,
            repetition_penalty=rep_penalty,
            generated_tokens=tokens # Pass history for penalty
        )

        next_token_int = int(next_token)
        tokens.append(next_token_int)

        # Decode and print just the new token for a cool streaming effect!
        new_word = tokenizer.decode([next_token_int])
        print(new_word, end="", flush=True)

    print() # Newline at the end
    return tokens

# # --- Test the new generation ---
# print("\n--- ADVANCED GENERATION ---")
# sample_key = jrandom.PRNGKey(999)
# _ = generate_text_advanced(
#     model,
#     "O Romeo, Romeo!",
#     tokenizer,
#     sample_key,
#     max_new_tokens=50,
#     temperature=0.85, # Slightly creative
#     top_k=50,         # Consider top 50 words
#     rep_penalty=1.2   # Mildly penalize repeating words
# )

#Part 3: Unsupervised Pre-Training

We optimize the model using Cross-Entropy on `train_tokens` and periodically evaluate it on `test_tokens` to ensure we are not memorizing the dataset.

The objective here is simple Next-Token Prediction. We use the Standard Cross-Entropy loss.

$$\mathscr{L} = -\frac{1}{N} \sum_{i} \log P(x_{i+1} | x_0, ..., x_i)$$

In [ ]:
def pretrain_loss_fn(model, batch_seq):
    inputs, targets = batch_seq[:-1], batch_seq[1:]
    logits = model(inputs)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets)
    return jnp.mean(loss)

@eqx.filter_value_and_grad
def compute_pretrain_loss_and_grads(model, batch_seq):
    return pretrain_loss_fn(model, batch_seq)

@eqx.filter_jit
def pretrain_step(model, opt_state, batch_seqs, optimizer):
    batch_loss_fn = jax.vmap(compute_pretrain_loss_and_grads, in_axes=(None, 0))
    losses, grads = batch_loss_fn(model, batch_seqs)
    grads = jax.tree_util.tree_map(lambda g: jnp.mean(g, axis=0), grads)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    return eqx.apply_updates(model, updates), opt_state, jnp.mean(losses)

@eqx.filter_jit
def evaluate_pretrain(model, batch_seqs):
    # Same as loss function, but no gradients calculated
    batch_loss_fn = jax.vmap(pretrain_loss_fn, in_axes=(None, 0))
    return jnp.mean(batch_loss_fn(model, batch_seqs))

# Optimizer and history
# Note: For larger models, a slightly lower learning rate (e.g., 1e-4) often
# yields more stable convergence than the 3e-4 used for tiny models.
optimizer = optax.adamw(learning_rate=1.5e-4, weight_decay=0.1)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

# We can increase the batch size to utilize the remaining 30GB of VRAM
epochs = 1000       # More epochs needed for a larger model to converge
batch_size = 64     # Doubled batch size (was 32)
train_losses, test_losses = [], []

print(f"Starting Pre-Training on GPT-2 Small Architecture (Batch Size: {batch_size})...")
for epoch in range(epochs):
    key, subkey1, subkey2 = jrandom.split(key, 3)

    # Train step
    train_batch = get_batch(train_tokens, batch_size, SEQ_LEN, subkey1)
    model, opt_state, train_loss = pretrain_step(model, opt_state, train_batch, optimizer)

    # Eval step
    if epoch % 50 == 0:
        test_batch = get_batch(test_tokens, batch_size, SEQ_LEN, subkey2)
        test_loss = evaluate_pretrain(model, test_batch)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        print(f"Step {epoch:4d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

plot_losses(train_losses, test_losses, "Pre-Training Cross-Entropy Loss")

print("\n--- ADVANCED GENERATION ---")
sample_key = jrandom.PRNGKey(999)
_ = generate_text_advanced(
    model,
    "O Romeo, Romeo!",
    tokenizer,
    sample_key,
    max_new_tokens=50,
    temperature=0.85, # Slightly creative
    top_k=50,         # Consider top 50 words
    rep_penalty=1.2   # Mildly penalize repeating words
)

In [ ]:
# ==========================================
# SAVE / LOAD PRE-TRAINED BASELINE
# ==========================================
pretrain_ckpt_path = f"{ckpt_dir}/model_pretrained.eqx"

# SAVE:
eqx.tree_serialise_leaves(pretrain_ckpt_path, model)
print(f"Pre-trained weights successfully saved to {pretrain_ckpt_path}")

# ---------------------------------------------------------
# HOW TO RECOVER IF COLAB CRASHES HERE:
# 1. Run Part 1 (Dataset) and Part 2 (Architecture).
# 2. Uncomment the line below and run this cell:
# model = eqx.tree_deserialise_leaves(pretrain_ckpt_path, model)
# print("Pre-trained model recovered!")
# ---------------------------------------------------------

#Part 4: Supervised Fine-Tuning (SFT)

A base model completes text. SFT transforms it into an assistant by training it on `[Prompt] + [Response]` pairs.

**Crucial SFT Logic**: We compute the forward pass on the entire sequence, but we set the loss multiplier to 0.0 for all tokens belonging to the Prompt. The model should not be penalized for failing to predict what the human asked; it should only be penalized for predicting the wrong answer.

In [ ]:
sft_prompts_train = ["Who is Romeo?", "Who is Juliet?", "What is Hamlet?"]
sft_responses_train = [" A Montague.", " A Capulet.", " The Prince."]

sft_prompts_test = ["Who is Tybalt?", "What is Macbeth?"]
sft_responses_test = [" Juliet's cousin.", " A Scottish King."]

def prepare_sft_batch(prompts, responses, tokenizer, seq_len):
    batch_seqs, batch_masks = [], []
    for p, r in zip(prompts, responses):
        p_tokens = tokenizer.encode(p)
        r_tokens = tokenizer.encode(r) + [tokenizer.eot_token]
        full_seq = p_tokens + r_tokens
        pad_len = max(0, (seq_len + 1) - len(full_seq))
        full_seq = (full_seq + [tokenizer.eot_token] * pad_len)[:seq_len + 1]

        # 0 for prompt, 1 for response
        mask = [0.0] * len(p_tokens) + [1.0] * (len(full_seq) - len(p_tokens))
        batch_seqs.append(full_seq); batch_masks.append(mask)
    return jnp.array(batch_seqs), jnp.array(batch_masks)

def sft_loss_fn(model, seq, mask):
    inputs, targets, target_mask = seq[:-1], seq[1:], mask[1:]
    logits = model(inputs)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets)
    return jnp.sum(loss * target_mask) / (jnp.sum(target_mask) + 1e-8)

@eqx.filter_value_and_grad
def compute_sft_loss(model, seq, mask): return sft_loss_fn(model, seq, mask)

@eqx.filter_jit
def sft_step(model, opt_state, seqs, masks, optimizer):
    losses, grads = jax.vmap(compute_sft_loss, in_axes=(None, 0, 0))(model, seqs, masks)
    grads = jax.tree_util.tree_map(lambda g: jnp.mean(g, axis=0), grads)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    return eqx.apply_updates(model, updates), opt_state, jnp.mean(losses)

@eqx.filter_jit
def eval_sft(model, seqs, masks):
    return jnp.mean(jax.vmap(sft_loss_fn, in_axes=(None, 0, 0))(model, seqs, masks))

# Prepare SFT Data
train_sft_seqs, train_sft_masks = prepare_sft_batch(sft_prompts_train, sft_responses_train, tokenizer, SEQ_LEN)
test_sft_seqs, test_sft_masks = prepare_sft_batch(sft_prompts_test, sft_responses_test, tokenizer, SEQ_LEN)

sft_optimizer = optax.adamw(learning_rate=1e-4)
sft_opt_state = sft_optimizer.init(eqx.filter(model, eqx.is_array))

sft_train_hist, sft_test_hist = [], []
print("Starting SFT...")
for epoch in range(150):
    model, sft_opt_state, tr_loss = sft_step(model, sft_opt_state, train_sft_seqs, train_sft_masks, sft_optimizer)

    if epoch % 10 == 0:
        te_loss = eval_sft(model, test_sft_seqs, test_sft_masks)
        sft_train_hist.append(tr_loss)
        sft_test_hist.append(te_loss)
        if epoch % 30 == 0:
            print(f"SFT Epoch {epoch:3d} | Train Loss: {tr_loss:.4f} | Test Loss: {te_loss:.4f}")

plot_losses(sft_train_hist, sft_test_hist, "SFT Masked Loss")

print("\n--- ADVANCED GENERATION ---")
sample_key = jrandom.PRNGKey(999)
_ = generate_text_advanced(
    model,
    "O Romeo, Romeo!",
    tokenizer,
    sample_key,
    max_new_tokens=50,
    temperature=0.85, # Slightly creative
    top_k=50,         # Consider top 50 words
    rep_penalty=1.2   # Mildly penalize repeating words
)

In [ ]:
# ==========================================
# SAVE / LOAD SFT MODEL
# ==========================================
sft_ckpt_path = f"{ckpt_dir}/model_sft.eqx"

# SAVE:
eqx.tree_serialise_leaves(sft_ckpt_path, model)
print(f"SFT weights successfully saved to {sft_ckpt_path}")

# ---------------------------------------------------------
# HOW TO RECOVER IF COLAB CRASHES HERE:
# 1. Run Part 1, Part 2, and the Checkpoint Setup.
# 2. Uncomment the line below to load the SFT weights:
# model = eqx.tree_deserialise_leaves(sft_ckpt_path, model)
# print("SFT model recovered!")
# ---------------------------------------------------------

#Part 5: Low-Rank Adaptation (LoRA)


Full fine-tuning (updating every weight) is computationally devastating for large models. LoRA solves this by freezing the pre-trained model weights and injecting trainable rank decomposition matrices into each layer.

For a pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, the modified forward pass is:

$$h = W_0 x + \Delta W x = W_0 x + B A x$$

Where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, and the rank $r \ll \min(d, k)$.


Using Equinox's filtering paradigm, we will replace the dense `lm_head` with a `LoRALinear` layer, and freeze the rest of the network.

In [ ]:

do_lora_training = False # Set this to False to recover saved chackpoint

class LoRALinear(eqx.Module):
    linear: eqx.nn.Linear
    lora_A: jax.Array
    lora_B: jax.Array
    scaling: float
    is_lora: bool = True

    def __init__(self, base_linear, r, alpha, key):
        self.linear = base_linear
        self.scaling = alpha / r
        k1, _ = jrandom.split(key)
        self.lora_A = jrandom.normal(k1, (r, base_linear.in_features)) * 0.01
        self.lora_B = jnp.zeros((base_linear.out_features, r))

    def __call__(self, x):
        return self.linear(x) + ((self.lora_B @ (self.lora_A @ x)) * self.scaling)

# Inject LoRA
key, subkey = jrandom.split(key)
lora_head = LoRALinear(model.lm_head, r=8, alpha=16, key=subkey)
lora_model = eqx.tree_at(lambda m: m.lm_head, model, lora_head)

# Isolate Trainable Parameters
def get_lora_filter(m):
    return lambda x: isinstance(x, jax.Array) and hasattr(x, "shape") and (
        x.shape == m.lm_head.lora_A.shape or x.shape == m.lm_head.lora_B.shape
    )

lora_optimizer = optax.adamw(learning_rate=1e-3)
trainable_init, _ = eqx.partition(lora_model, get_lora_filter(lora_model))
lora_opt_state = lora_optimizer.init(trainable_init)

@eqx.filter_jit
def lora_step(model, opt_state, seqs, masks, optimizer):
    trainable, frozen = eqx.partition(model, get_lora_filter(model))

    def loss_wrapper(t_part):
        comb_model = eqx.combine(t_part, frozen)
        return jnp.mean(jax.vmap(sft_loss_fn, in_axes=(None, 0, 0))(comb_model, seqs, masks))

    loss, grads = jax.value_and_grad(loss_wrapper)(trainable)
    updates, opt_state = optimizer.update(grads, opt_state, trainable)
    trainable = eqx.apply_updates(trainable, updates)
    return eqx.combine(trainable, frozen), opt_state, loss

if do_lora_training:
  lora_train_hist, lora_test_hist = [], []
  print("Starting LoRA Fine-Tuning...")
  for epoch in range(100):
      lora_model, lora_opt_state, tr_loss = lora_step(lora_model, lora_opt_state, train_sft_seqs, train_sft_masks, lora_optimizer)

      if epoch % 10 == 0:
          te_loss = eval_sft(lora_model, test_sft_seqs, test_sft_masks)
          lora_train_hist.append(tr_loss); lora_test_hist.append(te_loss)

  plot_losses(lora_train_hist, lora_test_hist, "LoRA Fine-Tuning Loss")

  print("\n--- ADVANCED GENERATION ---")
  sample_key = jrandom.PRNGKey(999)
  _ = generate_text_advanced(
      model,
      "O Romeo, Romeo!",
      tokenizer,
      sample_key,
      max_new_tokens=50,
      temperature=0.85, # Slightly creative
      top_k=50,         # Consider top 50 words
      rep_penalty=1.2   # Mildly penalize repeating words
  )

# print("\n--- GENERATION AFTER LORA ---")
# print(f"Prompt: 'Who is Juliet?'\nOutput: {generate_text(lora_model, 'Who is Juliet?', tokenizer, max_new_tokens=10)}")

In [ ]:
# ==========================================
# SAVE / LOAD LORA MODEL
# ==========================================
lora_ckpt_path = f"{ckpt_dir}/model_lora.eqx"

# SAVE:
eqx.tree_serialise_leaves(lora_ckpt_path, lora_model)
print(f"LoRA adapted weights successfully saved to {lora_ckpt_path}")

# ---------------------------------------------------------
# HOW TO RECOVER IF COLAB CRASHES HERE:
# 1. Run Parts 1 and 2 (Dataset and Base Architecture).
# 2. Run the first two cells of Part 5 to create the `lora_model` skeleton.
# 3. Uncomment the line below to load the LoRA weights:
lora_model = eqx.tree_deserialise_leaves(lora_ckpt_path, lora_model)
print("LoRA model recovered!")
# ---------------------------------------------------------

#Part 6: RLHF (PPO)

Finally, we align the policy. Because RL optimizes Reward rather than a static ground-truth test set, we plot the Mean Reward against the Actor Loss to evaluate convergence.

In [ ]:
import copy

class ValueModel(eqx.Module):
    embedding: eqx.nn.Embedding
    pos_embedding: eqx.nn.Embedding
    blocks: list
    ln_f: eqx.nn.LayerNorm
    value_head: eqx.nn.Linear

    def __init__(self, vocab_size, seq_len, d_model, num_heads, num_layers, key):
        keys = jrandom.split(key, num_layers + 2)
        self.embedding = eqx.nn.Embedding(vocab_size, d_model, key=keys[0])
        self.pos_embedding = eqx.nn.Embedding(seq_len, d_model, key=keys[1])
        self.blocks = [TransformerBlock(d_model, num_heads, key=keys[i+2]) for i in range(num_layers)]
        self.ln_f = eqx.nn.LayerNorm(d_model)
        self.value_head = eqx.nn.Linear(d_model, 1, key=keys[-1])

    def __call__(self, x):
        seq_len = x.shape[0]
        mask = jnp.where(jnp.tril(jnp.ones((seq_len, seq_len))) == 0, -jnp.inf, 0.0)
        positions = jnp.arange(seq_len)
        x = jax.vmap(self.embedding)(x) + jax.vmap(self.pos_embedding)(positions)
        for block in self.blocks: x = block(x, mask)
        x = jax.vmap(self.ln_f)(x)
        return jax.vmap(self.value_head)(x)[-1][0]

# ref_model = copy.deepcopy(model)
ref_model = model
key, subkey = jrandom.split(key)
critic = ValueModel(vocab_size, SEQ_LEN, D_MODEL, NUM_HEADS, NUM_LAYERS, subkey)

ppo_optimizer = optax.adamw(learning_rate=1e-5)
critic_optimizer = optax.adamw(learning_rate=5e-5)
ppo_opt_state = ppo_optimizer.init(eqx.filter(model, eqx.is_array))
critic_opt_state = critic_optimizer.init(eqx.filter(critic, eqx.is_array))

def get_log_probs(m, seq, prompt_len):
    logits, targets = m(seq[:-1]), seq[1:]
    log_probs = -optax.softmax_cross_entropy_with_integer_labels(logits, targets)
    return jnp.sum(log_probs[prompt_len - 1:])

def ppo_actor_loss_fn(actor_model, ref_model, seq, prompt_len, old_lp, advantage, beta=0.1):
    new_lp = get_log_probs(actor_model, seq, prompt_len)
    ref_lp = get_log_probs(ref_model, seq, prompt_len)

    kl_penalty = beta * (new_lp - ref_lp)
    adjusted_advantage = advantage - kl_penalty

    ratio = jnp.exp(new_lp - old_lp)
    unclipped = ratio * adjusted_advantage
    clipped = jnp.clip(ratio, 0.8, 1.2) * adjusted_advantage
    return -jnp.minimum(unclipped, clipped)

@eqx.filter_jit
def ppo_step(actor, critic, ref, a_opt_s, c_opt_s, seq, prompt_len, reward, old_lp, a_opt, c_opt):
    c_loss, c_grads = eqx.filter_value_and_grad(lambda c, s, r: jnp.square(c(s) - r))(critic, seq, reward)
    c_up, c_opt_s = c_opt.update(c_grads, c_opt_s, critic)
    critic = eqx.apply_updates(critic, c_up)

    advantage = reward - jax.lax.stop_gradient(critic(seq))

    a_loss, a_grads = eqx.filter_value_and_grad(ppo_actor_loss_fn)(actor, ref, seq, prompt_len, old_lp, advantage)
    a_up, a_opt_s = a_opt.update(a_grads, a_opt_s, actor)

    return eqx.apply_updates(actor, a_up), critic, a_opt_s, c_opt_s, a_loss, c_loss

# Simulated Rollout
dummy_prompt_len = 10
dummy_rollout_seqs = get_batch(train_tokens, batch_size=8, seq_len=SEQ_LEN, key=key)
# Simulate human preference: steadily increasing over epochs
mean_rewards_hist, actor_loss_hist = [], []


print("Starting PPO Rollout Optimization...")

for epoch in range(150):
    # 1. Get batch
    batch_seqs = get_batch(train_tokens, batch_size=4, seq_len=SEQ_LEN, key=key) # Smaller batch for memory
    dummy_rewards = jrandom.uniform(jrandom.PRNGKey(epoch), (4,), minval=0.0, maxval=0.5) + (epoch/150)*0.5

    # 2. Pre-calculate old log probs (This CAN be vmapped as it only returns arrays)
    # old_lps = jax.vmap(get_log_probs, in_axes=(None, 0, None))(model, batch_seqs, dummy_prompt_len)
    old_lps = jax.vmap(lambda seq: get_log_probs(model, seq, dummy_prompt_len))(batch_seqs)

    # 3. Sequential Update (Memory-safe and avoids the gelu/vmap error)
    epoch_a_loss = 0
    epoch_c_loss = 0

    for i in range(len(batch_seqs)):
        model, critic, ppo_opt_state, critic_opt_state, a_loss, c_loss = ppo_step(
            model, critic, ref_model, ppo_opt_state, critic_opt_state,
            batch_seqs[i], dummy_prompt_len, dummy_rewards[i], old_lps[i],
            ppo_optimizer, critic_optimizer
        )
        epoch_a_loss += a_loss
        epoch_c_loss += c_loss

    if epoch % 10 == 0:
        mean_rewards_hist.append(jnp.mean(dummy_rewards))
        actor_loss_hist.append(epoch_a_loss / len(batch_seqs))
        print(f"Epoch {epoch} | Actor Loss: {actor_loss_hist[-1]:.4f} | Reward: {mean_rewards_hist[-1]:.4f}")



# Plot PPO curves
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(actor_loss_hist, 'g-', label='Actor Loss (PPO Obj)')
ax2.plot(mean_rewards_hist, 'b--', label='Mean Human Reward')
ax1.set_xlabel('PPO Epochs (x10)')
ax1.set_ylabel('Actor Loss', color='g')
ax2.set_ylabel('Reward', color='b')
plt.title('RLHF PPO Alignment Progress')
plt.show()

print("\n--- ADVANCED GENERATION ---")
sample_key = jrandom.PRNGKey(999)
_ = generate_text_advanced(
    model,
    "O Romeo, Romeo!",
    tokenizer,
    sample_key,
    max_new_tokens=50,
    temperature=0.85, # Slightly creative
    top_k=50,         # Consider top 50 words
    rep_penalty=1.2   # Mildly penalize repeating words
)
# print("\n--- GENERATION AFTER RLHF (PPO) ---")
# print(f"Prompt: 'What is Hamlet?'\nOutput: {generate_text(model, 'What is Hamlet?', tokenizer, max_new_tokens=10)}")

In [ ]:
# ==========================================
# SAVE / LOAD RLHF FINAL MODEL
# ==========================================
rlhf_ckpt_path = f"{ckpt_dir}/model_rlhf_final.eqx"

# SAVE:
eqx.tree_serialise_leaves(rlhf_ckpt_path, model)
print(f"Final RLHF aligned weights successfully saved to {rlhf_ckpt_path}")

# ---------------------------------------------------------
# HOW TO RECOVER IF COLAB CRASHES HERE:
# 1. Run Part 1 (Dataset) and Part 2 (Architecture).
# 2. Uncomment the line below to load the final aligned weights:
# model = eqx.tree_deserialise_leaves(rlhf_ckpt_path, model)
# print("RLHF model recovered! Ready for inference.")
# ---------------------------------------------------------

In [ ]:
print("\n--- ADVANCED GENERATION ---")
sample_key = jrandom.PRNGKey(999)
_ = generate_text_advanced(
    model,
    "O Romeo, Romeo!",
    tokenizer,
    sample_key,
    max_new_tokens=50,
    temperature=0.85, # Slightly creative
    top_k=50,         # Consider top 50 words
    rep_penalty=1.2   # Mildly penalize repeating words
)

#Part 7: Scaling Up to LLaMA and Gemma


Congratulations! The model you just built and trained is functionally identical to the architecture of GPT-2. However, the open-weight revolution led by models like Meta's LLaMA 1/2/3 and Google's Gemma introduced several critical architectural upgrades to maximize training stability and inference efficiency.If you wanted to scale this CSCI 6366 project up to a 7-Billion parameter frontier model, you would need to implement the following five upgrades:

1. Rotary Positional Embeddings (RoPE)

> * **What we used:** Learned Absolute Positional Embeddings (eqx.nn.Embedding). We literally added a learned vector to the token embedding based on its absolute index $t$.
> * The Modern Way: RoPE encodes position by rotating the Query and Key vectors in the attention mechanism using a rotation matrix.Why? It allows the model to extrapolate to longer sequence lengths than it was trained on, and it naturally captures relative distances between tokens better than absolute position vectors.2. RMSNorm (Root Mean Square Normalization)What we used: Standard LayerNorm, which centers the activations (subtracts the mean) and scales them (divides by variance), plus learned biases.The Modern Way: RMSNorm removes the mean-centering step entirely and drops the bias vectors.Why? Removing the mean-centering calculation yields a ~10-40% speedup in normalization operations without any degradation in model performance.3. SwiGLU / GeGLU ActivationsWhat we used: A standard MLP with a GELU activation.The Modern Way: Gated Linear Units (GLUs). Instead of a single linear projection followed by an activation, the input is projected twice. One projection passes through a Swish (SiLU) or GELU activation, and is then element-wise multiplied by the other linear projection.Why? Empirical scaling laws show that gated activations yield significantly better perplexity for a given parameter count compared to standard MLPs.4. Grouped-Query Attention (GQA)What we used: Multi-Head Attention (MHA). Every attention head has its own unique Query, Key, and Value matrices.The Modern Way: GQA groups multiple Query heads together to share a single Key and Value head.Why? KV-Cache Memory. During generation, the model must cache past Keys and Values to avoid recomputing them. For a 7B model with an 8,000 token context window, a standard MHA KV-cache requires massive amounts of VRAM. GQA drastically reduces memory overhead, allowing for faster inference and larger batch sizes.5. Infrastructure: JAX pjit and FSDPWhat we used: A model that fits entirely on a single GPU.The Modern Way: A 7B model requires ~14GB of VRAM just to store the weights in bfloat16, plus optimizer states and activations. To train it, you must shard the model across hundreds of GPUs using Fully Sharded Data Parallel (FSDP). In JAX, this is done elegantly using jax.sharding.Mesh and jax.jit to automatically partition the PyTree across TPU or GPU pods.